<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/nbeats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# N-BEATS

N-BEATS is a deep learning model for forecasting. It's a stack of fully-connected blocks,
where each block looks at the input window and produces two things: a "backcast" (its attempt
to explain the input) and a forecast. The backcast is subtracted before the next block, so
every block works on what the previous ones couldn't explain, and all the block forecasts are
added up into the final prediction.

We don't use our lag features here. The model reads the raw history window of each series by
itself, so it basically learns its own lags. We just reshape the data into one sequence per
(Store, Dept) - neuralforecast wants a "long format" with unique_id / ds / y columns - and
train one single global model on all ~3300 series together. That's the interesting part:
series with little data can benefit from patterns learned on the bigger ones.

The forecast horizon is 39 weeks, which covers the whole validation block in one shot (and
after refitting on full train, the whole test block). To keep the "pipeline that runs on the
raw test set" requirement, the model is wrapped in a small sklearn-style class.

In [2]:
# ══ Bootstrap ══
!pip -q install mlflow kaggle neuralforecast

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
print("MLflow →", setup_env())

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)

MLflow → https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow
train: (421570, 16) | test: (115064, 15)


In [3]:
import torch
assert torch.cuda.is_available(), \
    "no GPU - Runtime → Change runtime type → T4 GPU, then re-run from the top"
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [4]:
import time
import numpy as np, pandas as pd
import mlflow, mlflow.sklearn
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline

from src.train_val_split import time_based_split
from src.metrics import wmae

# last 39 weeks of train as validation - same length as the real test horizon
VAL_WEEKS = 39
tr, va = time_based_split(train, val_weeks=VAL_WEEKS)
Xtr, ytr = tr.drop(columns=["Weekly_Sales"]), tr["Weekly_Sales"]
Xva, yva = va.drop(columns=["Weekly_Sales"]), va["Weekly_Sales"]
X_full, y_full = train.drop(columns=["Weekly_Sales"]), train["Weekly_Sales"]
print(f"train fold {tr.Date.min().date()}..{tr.Date.max().date()} ({len(tr):,} rows)")
print(f"val fold   {va.Date.min().date()}..{va.Date.max().date()} "
      f"({len(va):,} rows, {int(va.IsHoliday.sum()):,} holiday rows)")

train fold 2010-02-05..2012-01-27 (305,982 rows)
val fold   2012-02-03..2012-10-26 (115,588 rows, 5,967 holiday rows)


In [5]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE


class NBeatsRegressor(BaseEstimator, RegressorMixin):
    '''Wraps neuralforecast's NBEATS so it behaves like an sklearn regressor.

    fit reshapes (Store, Dept, Date) + y into the long format and trains one
    global model over every series that is long enough (needs at least
    input_size + h points to form a training window).
    predict forecasts h weeks past the training history and joins the result
    back onto the rows of X. Rows we have no forecast for (too-short series,
    unseen keys) get the series' train mean, or the global median as a last
    resort - the covered fraction is printed so nothing fails silently.
    '''

    def __init__(self, h=39, input_size=52, max_steps=1500,
                 learning_rate=1e-3, random_seed=42):
        self.h = h
        self.input_size = input_size
        self.max_steps = max_steps
        self.learning_rate = learning_rate
        self.random_seed = random_seed

    @staticmethod
    def _long(X):
        return pd.DataFrame({
            "unique_id": X["Store"].astype(str) + "_" + X["Dept"].astype(str),
            "ds": X["Date"].to_numpy(),
        })

    def fit(self, X, y):
        df = self._long(X)
        df["y"] = np.asarray(y, dtype=float)
        df = df.sort_values(["unique_id", "ds"])
        counts = df.groupby("unique_id")["y"].size()
        min_len = self.input_size + self.h
        keep = counts[counts >= min_len].index
        print(f"training on {len(keep)} series; "
              f"{len(counts) - len(keep)} shorter than {min_len} obs -> mean fallback")
        self.series_mean_ = df.groupby("unique_id")["y"].mean()
        self.global_median_ = float(df["y"].median())
        model = NBEATS(h=self.h, input_size=self.input_size, loss=MAE(),
                       max_steps=self.max_steps, learning_rate=self.learning_rate,
                       random_seed=self.random_seed)
        self.nf_ = NeuralForecast(models=[model], freq="W-FRI")
        self.nf_.fit(df=df[df["unique_id"].isin(keep)])
        return self

    def predict(self, X):
        fcst = self.nf_.predict()
        if "unique_id" not in fcst.columns:
            fcst = fcst.reset_index()
        fcst = fcst.rename(columns={"NBEATS": "yhat"})
        out = self._long(X).merge(fcst[["unique_id", "ds", "yhat"]],
                                  on=["unique_id", "ds"], how="left")
        missing = out["yhat"].isna()
        out.loc[missing, "yhat"] = out.loc[missing, "unique_id"].map(self.series_mean_)
        out["yhat"] = out["yhat"].fillna(self.global_median_)
        print(f"forecast coverage: {100 * (1 - missing.mean()):.1f}% direct, rest fallback")
        return out["yhat"].to_numpy()

In [6]:
mlflow.set_experiment("NBEATS_Training")
RESULTS = {}

def run_nbeats(run_name, **kw):
    with mlflow.start_run(run_name=run_name):
        pipe = Pipeline([("model", NBeatsRegressor(**kw))])
        t0 = time.time(); pipe.fit(Xtr, ytr)
        score = wmae(yva, pipe.predict(Xva), Xva["IsHoliday"])
        mlflow.log_params(pipe.named_steps["model"].get_params())
        mlflow.log_metrics({"val_wmae": score, "fit_seconds": time.time() - t0})
        RESULTS[run_name] = (score, kw)
        print(f"{run_name:24s} val WMAE {score:9.1f}")
        return score

# run 1: one-year window, so every training window sees a full seasonal cycle.
# min series length becomes 52 + 39 = 91 of the ~104 weeks in the train fold
run_nbeats("nbeats_input52", input_size=52)

2026/07/09 09:53:45 INFO mlflow.tracking.fluent: Experiment with name 'NBEATS_Training' does not exist. Creating a new experiment.
INFO:lightning_fabric.utilities.seed:Seed set to 42


training on 2798 series; 508 shorter than 91 obs -> mean fallback


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
7.2 K     Non-trainable params
2.6 M     Total params
10.321    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/tmp/ipykernel_716/622443277.py:59: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[9034.6726506 9034.6726506 9034.6726506 ...  159.                 nan
          nan]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  out.loc[missing, "yhat"] = out.loc[missing, "unique_id"].map(self.series_mean_)


forecast coverage: 93.9% direct, rest fallback
nbeats_input52           val WMAE    1685.9
🏃 View run nbeats_input52 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/3/runs/df08a0ed7ebb43b5b66afea5ce953cc2
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/3


1685.8616625986085

In [7]:
# run 2: half-year window - more training samples and it covers sparser series
# (min length drops to 65), but a single window never sees a full yearly cycle
run_nbeats("nbeats_input26", input_size=26)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.5 M  | train
-------------------------------------------------------
2.5 M     Trainable params
5.1 K     Non-trainable params
2.5 M     Total params
10.100    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


training on 2906 series; 400 shorter than 65 obs -> mean fallback


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=1500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/tmp/ipykernel_716/622443277.py:59: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[  23.70153846 -166.25       -166.25       ...  159.                   nan
           nan]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  out.loc[missing, "yhat"] = out.loc[missing, "unique_id"].map(self.series_mean_)


forecast coverage: 96.5% direct, rest fallback
nbeats_input26           val WMAE    1929.7
🏃 View run nbeats_input26 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/3/runs/1260d5b531cc411c9cfd906af380939e
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/3


1929.6781181347928

In [8]:
# refit the better config on the whole train set and log the final pipeline
best_name = min(RESULTS, key=lambda k: RESULTS[k][0])
score, best_kw = RESULTS[best_name]
print("refitting on full train:", best_name, best_kw)

final_pipe = Pipeline([("model", NBeatsRegressor(**best_kw))])
with mlflow.start_run(run_name=f"final_full_train__{best_name}"):
    t0 = time.time(); final_pipe.fit(X_full, y_full)
    preds = final_pipe.predict(test)
    assert np.isfinite(preds).all()
    mlflow.log_params({"chosen_run": best_name, "refit_on": "full_train", **best_kw})
    mlflow.log_metrics({"val_wmae_of_chosen_cfg": score,
                        "fit_seconds": time.time() - t0})
    try:
        mlflow.sklearn.log_model(final_pipe, "model",
                             serialization_format="cloudpickle")
    except Exception as e:   # torch models sometimes refuse to pickle
        print("log_model failed, saving native checkpoint instead:", e)
        final_pipe.named_steps["model"].nf_.save("nbeats_ckpt", overwrite=True)
        mlflow.log_artifacts("nbeats_ckpt", "model_ckpt")
print("sample test preds:", preds[:3].round(1))

refitting on full train: nbeats_input52 {'input_size': 52}


INFO:lightning_fabric.utilities.seed:Seed set to 42


training on 2900 series; 431 shorter than 91 obs -> mean fallback


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
7.2 K     Non-trainable params
2.6 M     Total params
10.321    Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

/tmp/ipykernel_716/622443277.py:59: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[23.99507042 23.99507042 23.99507042 ... 44.70569231 44.70569231
 44.70569231]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  out.loc[missing, "yhat"] = out.loc[missing, "unique_id"].map(self.series_mean_)


forecast coverage: 96.8% direct, rest fallback


2026/07/09 09:54:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/09 09:54:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/09 09:55:16 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


🏃 View run final_full_train__nbeats_input52 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/3/runs/bb0b45d29d9b4ddd95e6507062771f67
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/3
sample test preds: [29916.8 20360.9 20199. ]


One thing to keep an eye on is the printed "forecast coverage" line: series that were too
short to train on just get their mean as the prediction, and that fraction limits how much of
the score the network actually controls. Otherwise this is the cleanest contrast to the tree
model - zero feature engineering, the network only sees raw sequences.